# Bag of Words (BoW) Encoding

**Goal:** preprocess raw text, build a vocabulary, and represent each document as a word-frequency vector.

**Flow:** Raw text → Clean & lemmatize → Build vocabulary → Count word frequencies → BoW matrix

## Step 1: Import tools

- `CountVectorizer` handles vocabulary building and word counting in one step.
- `nltk` provides tokenization, stopword lists, and lemmatization for text cleaning.
- `re` is used for regex-based punctuation removal.
- `pandas` is available for optional DataFrame inspection later.

In [2]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import re 
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
print('All imports successful')

All imports successful


## Step 2: Download NLTK resources

- `punkt` — tokenizer model for splitting text into words.
- `stopwords` — common English words to filter out (e.g., "I", "the", "is").
- `wordnet` — lexical database used by the lemmatizer to reduce words to base form.

In [3]:
nltk.download('punkt_tab', quiet=True)   # newer NLTK versions need punkt_tab
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
print("NLTK resources ready.")

NLTK resources ready.


## Step 3: Build the preprocessing pipeline

- Lowercasing normalizes case (`DEep` → `deep`).
- Regex strips punctuation and numbers so only alphabetic tokens remain.
- Stopword removal drops high-frequency, low-information words.
- Lemmatization collapses inflected forms (`bugs` → `bug`).

**Why this order?** Lowercasing first avoids case mismatches in the stopword list. Lemmatization runs last so it operates on clean tokens.

**Why `set()` for stopwords?** Lookup in a `set` is O(1) vs O(n) in a `list` — matters when checking every token against ~180 stopwords.

**Why a function?** Wrapping the pipeline in `clean_text()` keeps it reusable and testable. Each sentence goes through the same steps consistently.

**Alternative:** skip manual cleaning and pass a custom `preprocessor` or `tokenizer` to `CountVectorizer` directly.

In [4]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()                       # lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # remove punctuation/numbers
    tokens = word_tokenize(text)             # tokenize
    tokens = [word for word in tokens if word not in stop_words]   # remove stopwords
    tokens = [lemmatizer.lemmatize(word) for word in tokens]       # lemmatize
    return " ".join(tokens)

### Step 3 — Alternative: Let CountVectorizer handle preprocessing

Instead of a separate `clean_text()` function, you can pass a custom `tokenizer` directly into `CountVectorizer`. This bundles the entire pipeline (clean + tokenize + lemmatize) into the vectorizer itself.

**When to use this:** quick prototyping where you don't need the cleaned text separately.

**Trade-off:** cleaning is now tightly coupled to the vectorizer — you can't inspect or reuse the cleaned output independently.

In [7]:
# Alternative: custom tokenizer passed directly to CountVectorizer
def custom_tokenizer(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return tokens

# Keep this cell runnable even if executed before the sample-data cell
sample_doc = [
    "SDE loves coding",
    "SDE loves AI",
    "SDE codes in Python",
    "SDE codes AI"
]

# No need to pre-clean — the vectorizer calls custom_tokenizer internally
alt_vectorizer = CountVectorizer(tokenizer=custom_tokenizer)
alt_matrix = alt_vectorizer.fit_transform(sample_doc)

print("Alt vocabulary:", alt_vectorizer.get_feature_names_out())
print("Alt BoW matrix:\n", alt_matrix.toarray())
print("\n✅ Same result, zero manual preprocessing needed.")

d:\GenAI-Bootcamp-Practical\myenv\Lib\site-packages\sklearn\feature_extraction\text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Alt vocabulary: ['ai' 'code' 'coding' 'love' 'python' 'sde']
Alt BoW matrix:
 [[0 0 1 1 0 1]
 [1 0 0 1 0 1]
 [0 1 0 0 1 1]
 [1 1 0 0 0 1]]

✅ Same result, zero manual preprocessing needed.


## Step 4: Define sample data and clean it

- `doc` is the raw input corpus — short sentences with mixed case, numbers, and punctuation.
- Each sentence is passed through `clean_text()` to produce a normalized version.
- The cleaned output is what `CountVectorizer` will work with.

**Why a list comprehension?** Clean, Pythonic, and applies the same transform to every document in one line. A loop would work but is more verbose for no gain.

**Why clean before vectorizing?** `CountVectorizer` has built-in lowercasing and tokenization, but it doesn't handle lemmatization or custom stopword logic. Preprocessing externally gives full control over each step.

In [8]:
doc = [
    "SDE loves coding",
    "SDE loves AI",
    "SDE codes in Python",
    "SDE codes AI"
]
cleaned_doc = [clean_text(sentence) for sentence in doc]

# Side-by-side comparison: raw vs cleaned
print("Raw → Cleaned:")
for raw, clean in zip(doc, cleaned_doc):
    print(f"  {raw!r:40s} → {clean!r}")

Raw → Cleaned:
  'SDE loves coding'                       → 'sde love coding'
  'SDE loves AI'                           → 'sde love ai'
  'SDE codes in Python'                    → 'sde code python'
  'SDE codes AI'                           → 'sde code ai'


## Step 5: Initialize and fit the vectorizer

- `CountVectorizer()` builds a vocabulary from the corpus and counts word occurrences per document.
- `fit_transform()` does both in one call — learns the vocabulary and returns the sparse BoW matrix.

**Why `CountVectorizer` over a manual dict-based approach?** It handles tokenization, vocabulary building, and frequency counting internally with optimized sparse storage. A manual approach (looping tokens, building a `{word: index}` dict, filling a NumPy matrix) gives the same result but is slower and error-prone for large corpora.

**Why not pass preprocessing into the vectorizer?** We could set `stop_words='english'` and a custom `tokenizer`, but that tightly couples cleaning with vectorization. Keeping them separate lets you swap either independently.

**Alternative:** `TfidfVectorizer` — same interface but weights words by importance (TF-IDF) instead of raw counts. Drop-in replacement.

In [9]:
# Default settings: lowercase=True, token_pattern splits on word boundaries
vectorizer = CountVectorizer()

In [10]:
# Learn vocabulary and transform documents into a sparse BoW matrix
bow_matrix = vectorizer.fit_transform(cleaned_doc)
bow_matrix

<4x6 sparse matrix of type '<class 'numpy.int64'>'
	with 12 stored elements in Compressed Sparse Row format>

## Step 6: Inspect the BoW matrix

- `.toarray()` converts the sparse matrix to a dense NumPy array for readability.
- Each row = one document, each column = one vocabulary word.
- Cell values are raw word counts (0 if absent, 1+ if present).

**Why sparse by default?** Most documents use only a small subset of the full vocabulary. A 10k-word vocabulary with 20-word documents means ~99.8% zeros. Sparse format (CSR) stores only non-zero entries, saving memory dramatically.

**Trade-off:** `.toarray()` is fine for small demos but will blow up memory on real corpora. In production, work directly with the sparse matrix.

In [11]:
print(f"BoW representation:\n{bow_matrix.toarray()}")

BoW representation:
[[0 0 1 1 0 1]
 [1 0 0 1 0 1]
 [0 1 0 0 1 1]
 [1 1 0 0 0 1]]


## Step 7: Inspect the vocabulary

- `get_feature_names_out()` returns the list of words the vectorizer learned.
- Column order in the BoW matrix matches this vocabulary order.
- Useful for mapping matrix columns back to actual words.

**Alternative:** build a manual vocabulary using `set()` or a `Counter` over all tokens, then sort alphabetically to match `CountVectorizer`'s default behavior.

In [12]:
vectorizer.get_feature_names_out()

array(['ai', 'code', 'coding', 'love', 'python', 'sde'], dtype=object)

In [13]:
print(f"Vocabulary:\n{vectorizer.get_feature_names_out()}")

Vocabulary:
['ai' 'code' 'coding' 'love' 'python' 'sde']


- **Bag of Words** represents each document as a vector of word counts over a shared vocabulary.
- Text is preprocessed (lowercase → remove punctuation → tokenize → remove stopwords → lemmatize) before vectorization.
- `CountVectorizer` handles vocabulary building and counting; preprocessing is done externally for finer control.
- The output is a **sparse matrix** — convert with `.toarray()` to view the dense form.
- `fit_transform()` learns vocabulary and encodes in one step; use `transform()` alone for new unseen documents.
- BoW ignores word order — `"I love coding"` and `"coding love I"` produce the same vector.

**Limitations:** no word order, no semantic meaning, high-dimensional for large vocabularies.


1. **"What happens if a new unseen word appears at test time?"**
   `CountVectorizer.transform()` silently ignores it — the word won't have a column in the matrix. The vocabulary is fixed at `fit()` time. This means OOV (out-of-vocabulary) words are lost entirely.

2. **"Why is the BoW matrix sparse?"**
   Each document uses only a few words from the full vocabulary. With thousands of unique words, most entries are zero. Storing them all dense wastes memory — CSR sparse format only tracks non-zero values.

3. **"How does this scale for a corpus with 1M documents and 100K unique words?"**
   The matrix becomes 1M × 100K. Dense storage = ~400 GB (float32). Sparse format handles it because most entries are zero. But vocabulary size still grows — use `max_features` or `min_df`/`max_df` to cap it.

4. **"Can BoW preserve word order?"**
   No. `"dog bites man"` and `"man bites dog"` produce identical vectors. To capture order, use n-grams (`ngram_range=(1,2)`) or sequence models (RNN, Transformers).

5. **"Why lemmatize instead of stem?"**
   Lemmatization produces valid dictionary words (`better` → `good`), while stemming often gives fragments (`studies` → `studi`). Lemmatization is slower but more meaningful for downstream tasks.

6. **"What if two documents have the same words but different frequencies?"**
   BoW will produce different vectors — that's the point. But if raw counts dominate (one word appears 100x), it distorts similarity. TF-IDF normalizes this by down-weighting frequent terms.

7. **"Why use `set()` for stopwords instead of a list?"**
   `set` gives O(1) membership check. With ~180 stopwords checked against every token in the corpus, this adds up. A `list` would be O(n) per lookup.

8. **"What's the difference between `fit_transform()` and calling `fit()` then `transform()` separately?"**
   Same result, but `fit_transform()` is optimized internally — it avoids a redundant pass over the data. Use separate calls when you fit on training data and transform test data later.

1. **"If two sentences have the exact same words but different meanings, how will BoW behave?"**
   Identically. `"The bank of the river"` and `"I went to the bank"` produce the same vector for overlapping words. BoW has **no sense of semantics or context** — it's purely lexical. You'd need word embeddings (Word2Vec, BERT) to distinguish meaning.

2. **"How would you modify this pipeline to capture word importance, not just counts?"**
   Replace `CountVectorizer` with `TfidfVectorizer`. TF-IDF down-weights words that appear in many documents (like `"the"`, `"is"`) and boosts rare, informative terms. Same API, one-line swap.

3. **"Your vocabulary will explode on real data. How do you control it?"**
   Use `max_features=N` to keep only top-N words, or `min_df` / `max_df` to filter by document frequency. For example, `max_df=0.9` drops words appearing in >90% of docs (likely stopwords the list missed).

4. **"BoW gives you a fixed-size vector. What if documents vary wildly in length?"**
   The vector length is always `vocab_size` regardless of document length. But longer docs naturally have higher counts. Normalize with L2-norm or use TF-IDF (which includes length normalization) to make vectors comparable.

5. **"Can you use BoW for similarity search between documents?"**
   Yes — compute cosine similarity between BoW vectors. But it only captures surface-level overlap. Two documents about the same topic using different synonyms will appear dissimilar. For semantic similarity, use dense embeddings instead.